In [1]:
import duckdb
import pandas as pd
from pathlib import Path

DATA_DIR = Path.cwd().parent / "data"

con = duckdb.connect("olist.duckdb")

tables = {
    "orders":      "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "customers":   "olist_customers_dataset.csv",
    "payments":    "olist_order_payments_dataset.csv",
    "reviews":     "olist_order_reviews_dataset.csv",
}

for name, file in tables.items():
    path = (DATA_DIR / file).as_posix() 
    con.execute(f"CREATE OR REPLACE VIEW {name} AS SELECT * FROM read_csv_auto('{path}')")

df = con.execute("SELECT COUNT(*) AS total_orders FROM orders").df()
print(df)


   total_orders
0         99441


In [2]:
df_revenue = con.execute("""
    SELECT
        DATE_TRUNC('month', order_purchase_timestamp::TIMESTAMP) AS month,
        COUNT(DISTINCT o.order_id) AS total_orders,   -- distinct orders, not line items
        ROUND(SUM(oi.price + oi.freight_value), 2) AS revenue
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY month
    ORDER BY month
""").df()

print(df_revenue)


        month  total_orders     revenue
0  2016-09-01             1      143.46
1  2016-10-01           265    46490.66
2  2016-12-01             1       19.62
3  2017-01-01           750   127482.37
4  2017-02-01          1653   271239.32
5  2017-03-01          2546   414330.95
6  2017-04-01          2303   390812.40
7  2017-05-01          3546   566851.40
8  2017-06-01          3135   490050.37
9  2017-07-01          3872   566299.08
10 2017-08-01          4193   645832.36
11 2017-09-01          4150   701077.49
12 2017-10-01          4478   751117.01
13 2017-11-01          7289  1153364.20
14 2017-12-01          5513   843078.29
15 2018-01-01          7069  1077887.46
16 2018-02-01          6555   966168.41
17 2018-03-01          7003  1120598.24
18 2018-04-01          6798  1132878.93
19 2018-05-01          6749  1128774.52
20 2018-06-01          6099  1011978.29
21 2018-07-01          6159  1027807.28
22 2018-08-01          6351   985491.64


In [3]:
tables = ['orders', 'order_items', 'customers', 'payments', 'reviews']

for table in tables: 
    count = con.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]
    print(f'{table}: {count:,} rows')

orders: 99,441 rows
order_items: 112,650 rows
customers: 99,441 rows


payments: 103,886 rows
reviews: 99,224 rows


In [4]:
for table in tables:
    print(table)
    total = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    cols = con.execute(f"SELECT * FROM {table} LIMIT 0").df().columns

    for col in cols:
        nulls = con.execute(f"SELECT COUNT(*) FROM {table} WHERE {col} IS NULL").fetchone()[0]
        if nulls != 0:
            pct = round(nulls / total * 100, 1)
            print(f"  {col}: {nulls:,} nulls ({pct}%)")


orders


  order_approved_at: 160 nulls (0.2%)
  order_delivered_carrier_date: 1,783 nulls (1.8%)
  order_delivered_customer_date: 2,965 nulls (3.0%)


order_items


customers


payments


reviews


  review_comment_title: 87,656 nulls (88.3%)
  review_comment_message: 58,247 nulls (58.7%)


In [5]:
total = con.execute("SELECT COUNT(*) FROM orders").fetchone()[0]
unique = con.execute("SELECT COUNT(DISTINCT order_id) FROM orders").fetchone()[0]

print(f"Total rows: {total:,}")
print(f"Unique order_ids: {unique:,}")
print(f"Duplicates: {total - unique:,}")


Total rows: 99,441
Unique order_ids: 99,441
Duplicates: 0


In [6]:
print('order items')
result = con.execute("""

    SELECT
                     
        COUNT(*) FILTER(WHERE price <= 0) AS zero_or_negative_price, 
        COUNT(*) FILTER(WHERE freight_value < 0) AS negative_freight,
        MIN(price) AS min_price, 
        MAX(price) AS max_price,
        ROUND(AVG(price), 2) AS avg_price
    FROM order_items
        
 """).df()
print(result)

order items
   zero_or_negative_price  negative_freight  min_price  max_price  avg_price
0                       0                 0       0.85     6735.0     120.65


In [7]:
print("payments")

result = con.execute("""
    SELECT 
        COUNT(*) FILTER(WHERE payment_value <= 0) AS zero_or_negative_payment, 
        MIN(payment_value) AS min_payment,
        MAX(payment_value) AS max_payment,
        ROUND(AVG(payment_value), 2) AS avg_payment
    FROM payments
""").df()
print(result)

payments
   zero_or_negative_payment  min_payment  max_payment  avg_payment
0                         9          0.0     13664.08        154.1


In [8]:
print("dates")

result = con.execute("""
    SELECT 
        COUNT(*) FILTER(WHERE order_purchase_timestamp > order_delivered_customer_date)  AS delivered_before_purchase, 
        COUNT(*) FILTER(WHERE order_purchase_timestamp > CURRENT_DATE) AS future_dates,
        MIN(order_purchase_timestamp) AS min_timestamp,
        MAX(order_purchase_timestamp) AS max_timestamp          
    FROM orders
""").df()

print(result)

dates


   delivered_before_purchase  future_dates       min_timestamp  \
0                          0             0 2016-09-04 21:15:19   

        max_timestamp  
0 2018-10-17 17:30:18  


In [9]:
print("status")

result = con.execute("""
    SELECT 
        order_status,
        COUNT(*) AS count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS pct
    
    FROM orders
    GROUP BY order_status
    ORDER BY count DESC
 """).df()

print(result)

status


  order_status  count   pct
0    delivered  96478  97.0
1      shipped   1107   1.1
2     canceled    625   0.6
3  unavailable    609   0.6
4     invoiced    314   0.3
5   processing    301   0.3
6      created      5   0.0
7     approved      2   0.0


# SQL Analysis Blocks

The EDA and data-cleaning cells above are kept as-is. The cells below add the
core analysis.

### Code review notes on the existing code

A few small things worth flagging in the cells above (left in place, improved
in the analysis blocks that follow):

- **First revenue query** (`df_revenue`) reports raw monthly revenue but no
  growth signal. Block 1 below rebuilds it with a CTE + `LAG()` window function
  to compute month-over-month (MoM) growth.
- **`order_purchase_timestamp::TIMESTAMP` cast** is redundant — `read_csv_auto`
  already infers the column as `TIMESTAMP`. Harmless, but unnecessary.
- **`AS total_orders`** vs. the `orders` column name used downstream — the
  exported tables standardise on `orders` for consistency with the dashboard.
- The null/anomaly checks build SQL via f-strings over a hard-coded table list.
  Fine for a trusted, fixed schema like this one; would need parameterisation
  if column names were user-supplied.

### Why `customer_unique_id` (not `customer_id`)

In the Olist schema `customer_id` is regenerated for **every order**, so a
returning buyer gets a new `customer_id` each time. Joining on `customer_id`
would make every customer look like a one-time buyer and force retention to
~0%. All customer-level analysis below (cohorts, repeat rate, RFM) keys on
`customers.customer_unique_id`, the stable per-person identifier.

In [10]:
# Export directory: the Streamlit app reads these pre-computed CSVs so it can
# run without the raw dataset (e.g. on Streamlit Cloud).
EXPORT_DIR = Path.cwd() / "exports"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
print("Exports ->", EXPORT_DIR)

Exports -> D:\work\DS\Projects\Olist\analytics\exports


## Block 1 — Monthly revenue with MoM growth

Delivered revenue per month, plus month-over-month growth via a CTE and the
`LAG()` window function.

In [11]:
revenue = con.execute("""
    WITH monthly AS (
        SELECT
            DATE_TRUNC('month', o.order_purchase_timestamp) AS month,
            COUNT(DISTINCT o.order_id) AS orders,
            ROUND(SUM(oi.price + oi.freight_value), 2) AS revenue
        FROM orders o
        JOIN order_items oi ON o.order_id = oi.order_id
        WHERE o.order_status = 'delivered'
        GROUP BY month
    )
    SELECT
        month,
        orders,
        revenue,
        LAG(revenue) OVER (ORDER BY month) AS prev_revenue,
        ROUND(
            (revenue - LAG(revenue) OVER (ORDER BY month))
            / LAG(revenue) OVER (ORDER BY month) * 100, 1
        ) AS mom_growth_pct
    FROM monthly
    ORDER BY month
""").df()

revenue.to_csv(EXPORT_DIR / "revenue_monthly.csv", index=False)

# Use the median MoM, not the mean: the 2016 ramp-up months have revenue near
# zero, so their MoM percentages are five-digit outliers that wreck the mean.
print(f"Total delivered revenue: R$ {revenue['revenue'].sum():,.2f}")
print(f"Median MoM growth:       {revenue['mom_growth_pct'].median():.1f}%")
revenue.tail()

Total delivered revenue: R$ 15,419,773.75
Median MoM growth:       7.8%


,month,orders,revenue,prev_revenue,mom_growth_pct
18,2018-04-01,6798,1132878.93,1120598.24,1.1
19,2018-05-01,6749,1128774.52,1132878.93,-0.4
20,2018-06-01,6099,1011978.29,1128774.52,-10.3
21,2018-07-01,6159,1027807.28,1011978.29,1.6
22,2018-08-01,6351,985491.64,1027807.28,-4.1


## Block 2 — Cohort retention analysis

Customers are grouped into monthly cohorts by their **first** delivered order
(`cohort_month`). For each later month we measure how many of that cohort
ordered again, expressed as a percentage of the cohort's original size.
Everything keys on `customer_unique_id`.

In [12]:
cohort = con.execute("""
    WITH cust_orders AS (
        SELECT
            c.customer_unique_id,
            DATE_TRUNC('month', o.order_purchase_timestamp) AS order_month
        FROM orders o
        JOIN customers c ON o.customer_id = c.customer_id
        WHERE o.order_status = 'delivered'
    ),
    cohorts AS (
        SELECT customer_unique_id, MIN(order_month) AS cohort_month
        FROM cust_orders
        GROUP BY customer_unique_id
    ),
    activity AS (
        SELECT
            ch.cohort_month,
            DATEDIFF('month', ch.cohort_month, co.order_month) AS month_offset,
            COUNT(DISTINCT co.customer_unique_id) AS customers
        FROM cust_orders co
        JOIN cohorts ch ON co.customer_unique_id = ch.customer_unique_id
        GROUP BY ch.cohort_month, month_offset
    ),
    sizes AS (
        SELECT cohort_month, customers AS cohort_size
        FROM activity WHERE month_offset = 0
    )
    SELECT
        a.cohort_month,
        a.month_offset,
        a.customers,
        s.cohort_size,
        ROUND(a.customers * 100.0 / s.cohort_size, 1) AS retention_pct
    FROM activity a
    JOIN sizes s ON a.cohort_month = s.cohort_month
    ORDER BY a.cohort_month, a.month_offset
""").df()
cohort.to_csv(EXPORT_DIR / "cohort_retention.csv", index=False)

# Overall repeat-purchase rate (share of customers with 2+ delivered orders).
repeat = con.execute("""
    WITH per_customer AS (
        SELECT c.customer_unique_id, COUNT(DISTINCT o.order_id) AS n_orders
        FROM orders o
        JOIN customers c ON o.customer_id = c.customer_id
        WHERE o.order_status = 'delivered'
        GROUP BY c.customer_unique_id
    )
    SELECT
        COUNT(*) AS customers,
        COUNT(*) FILTER (WHERE n_orders >= 2) AS repeat_customers,
        ROUND(COUNT(*) FILTER (WHERE n_orders >= 2) * 100.0 / COUNT(*), 1)
            AS repeat_rate_pct
    FROM per_customer
""").df()
repeat.to_csv(EXPORT_DIR / "repeat_rate.csv", index=False)

print(repeat.to_string(index=False))
# Retention as a pivot (cohort x month offset) for a quick visual sanity check.
cohort.pivot(index="cohort_month", columns="month_offset",
             values="retention_pct").iloc[:8, :6]

 customers  repeat_customers  repeat_rate_pct
     93358              2801              3.0


month_offset,0,1,2,3,4,5
cohort_month,,,,,,
2016-09-01,100.0,NaN,NaN,NaN,NaN,NaN
2016-10-01,100.0,NaN,NaN,NaN,NaN,NaN
2016-12-01,100.0,100.0,NaN,NaN,NaN,NaN
2017-01-01,100.0,0.3,0.3,0.1,0.4,0.1
2017-02-01,100.0,0.2,0.3,0.1,0.4,0.1
2017-03-01,100.0,0.4,0.4,0.4,0.4,0.2
2017-04-01,100.0,0.6,0.2,0.2,0.3,0.3
2017-05-01,100.0,0.5,0.5,0.3,0.3,0.3


## Block 3 — RFM segmentation

For each `customer_unique_id` we compute **Recency** (days since last order,
relative to the dataset's latest timestamp), **Frequency** (number of delivered
orders) and **Monetary** (total spend incl. freight). Recency and Monetary are
bucketed into quartiles with `NTILE(4)`.

Because Olist is overwhelmingly a single-purchase marketplace (Frequency is
~1 for almost everyone), segments are driven mainly by Recency and Monetary,
with repeat buyers promoted:

| Segment   | Rule |
|-----------|------|
| Champions | repeat buyer & recent, **or** recent & high spend |
| Loyal     | repeat buyer, **or** recent & lower spend |
| At Risk   | lapsed but high spend |
| Lost      | lapsed & low spend |

In [13]:
rfm = con.execute("""
    WITH bounds AS (
        SELECT MAX(order_purchase_timestamp) AS snapshot FROM orders
    ),
    per_customer AS (
        SELECT
            c.customer_unique_id,
            DATEDIFF('day', MAX(o.order_purchase_timestamp),
                     (SELECT snapshot FROM bounds)) AS recency_days,
            COUNT(DISTINCT o.order_id) AS frequency,
            ROUND(SUM(oi.price + oi.freight_value), 2) AS monetary
        FROM orders o
        JOIN customers c ON o.customer_id = c.customer_id
        JOIN order_items oi ON o.order_id = oi.order_id
        WHERE o.order_status = 'delivered'
        GROUP BY c.customer_unique_id
    ),
    scored AS (
        SELECT
            *,
            NTILE(4) OVER (ORDER BY recency_days DESC) AS r_score,  -- 4 = most recent
            NTILE(4) OVER (ORDER BY monetary ASC)     AS m_score   -- 4 = highest spend
        FROM per_customer
    ),
    segmented AS (
        SELECT
            *,
            CASE
                WHEN frequency >= 2 AND r_score >= 3 THEN 'Champions'
                WHEN frequency >= 2                  THEN 'Loyal'
                WHEN r_score >= 3 AND m_score >= 3   THEN 'Champions'
                WHEN r_score >= 3 AND m_score <  3   THEN 'Loyal'
                WHEN r_score <  3 AND m_score >= 3   THEN 'At Risk'
                ELSE 'Lost'
            END AS segment
        FROM scored
    )
    SELECT
        segment,
        COUNT(*) AS customers,
        ROUND(AVG(recency_days), 0) AS avg_recency_days,
        ROUND(AVG(frequency), 2) AS avg_frequency,
        ROUND(AVG(monetary), 2) AS avg_monetary,
        ROUND(SUM(monetary), 2) AS total_revenue,
        ROUND(SUM(monetary) * 100.0 / SUM(SUM(monetary)) OVER (), 1) AS pct_revenue
    FROM segmented
    GROUP BY segment
    ORDER BY total_revenue DESC
""").df()
rfm.to_csv(EXPORT_DIR / "rfm_segments.csv", index=False)
rfm

,segment,customers,avg_recency_days,avg_frequency,avg_monetary,total_revenue,pct_revenue
0,Champions,24019,160.0,1.07,264.63,6356152.39,41.2
1,At Risk,21735,413.0,1.00,264.36,5745923.18,37.3
2,Loyal,23917,173.0,1.06,75.51,1805885.82,11.7
3,Lost,23687,413.0,1.00,63.82,1511812.36,9.8


## Block 4 — Delivery funnel + late delivery vs. review score

The funnel tracks how orders progress through the fulfilment stages
(purchased → approved → shipped → delivered → delivered on time), using the
timestamp columns in `orders`. We then split **delivered** orders into
*on time* vs. *late* (delivered after `order_estimated_delivery_date`) and
compare average review scores — the motivation for the A/B test that follows.

In [14]:
funnel = con.execute("""
    SELECT 'Purchased' AS stage, COUNT(*) AS orders FROM orders
    UNION ALL
    SELECT 'Approved', COUNT(*) FROM orders WHERE order_approved_at IS NOT NULL
    UNION ALL
    SELECT 'Shipped', COUNT(*) FROM orders
        WHERE order_delivered_carrier_date IS NOT NULL
    UNION ALL
    SELECT 'Delivered', COUNT(*) FROM orders
        WHERE order_delivered_customer_date IS NOT NULL
    UNION ALL
    SELECT 'Delivered on time', COUNT(*) FROM orders
        WHERE order_delivered_customer_date IS NOT NULL
          AND order_delivered_customer_date <= order_estimated_delivery_date
""").df()
stage_order = ["Purchased", "Approved", "Shipped", "Delivered", "Delivered on time"]
funnel["stage"] = pd.Categorical(funnel["stage"], categories=stage_order, ordered=True)
funnel = funnel.sort_values("stage").reset_index(drop=True)
funnel.to_csv(EXPORT_DIR / "delivery_funnel.csv", index=False)

late_vs_score = con.execute("""
    SELECT
        CASE WHEN o.order_delivered_customer_date
                  > o.order_estimated_delivery_date
             THEN 'Late' ELSE 'On time' END AS delivery_status,
        COUNT(*) AS orders,
        ROUND(AVG(r.review_score), 3) AS avg_review_score,
        ROUND(MEDIAN(r.review_score), 1) AS median_review_score
    FROM orders o
    JOIN reviews r ON o.order_id = r.order_id
    WHERE o.order_status = 'delivered'
      AND o.order_delivered_customer_date IS NOT NULL
    GROUP BY delivery_status
    ORDER BY delivery_status
""").df()
late_vs_score.to_csv(EXPORT_DIR / "delivery_late_vs_score.csv", index=False)

print(funnel.to_string(index=False))
print()
print(late_vs_score.to_string(index=False))

            stage  orders
        Purchased   99441
         Approved   99281
          Shipped   97658
        Delivered   96476
Delivered on time   88649

delivery_status  orders  avg_review_score  median_review_score
           Late    7700             2.566                  2.0
        On time   88653             4.294                  5.0


## A/B test — does on-time delivery earn higher review scores?

**Hypothesis (H1):** orders delivered on time receive significantly higher
review scores than orders delivered late.

Review scores are an ordinal 1–5 scale and clearly non-normal (heavily skewed
toward 5), so a t-test is inappropriate. We use the **Mann-Whitney U** test
(one-sided: on-time > late) and report it together with an **effect size**, not
just the p-value:

- **Cliff's delta** — a non-parametric effect size for ordinal data, computed
  via the identity `δ = 2U / (n₁·n₂) − 1`. Magnitude thresholds (Romano et al.):
  negligible < 0.147, small < 0.33, medium < 0.474, large ≥ 0.474.
- **95% bootstrap CI** for Cliff's delta (1,000 resamples) so the effect comes
  with an uncertainty range.

In [15]:
import numpy as np
from scipy.stats import mannwhitneyu

# Pull review scores for delivered orders, tagged on-time vs late.
scores = con.execute("""
    SELECT
        CASE WHEN o.order_delivered_customer_date
                  > o.order_estimated_delivery_date
             THEN 'Late' ELSE 'On time' END AS delivery_status,
        r.review_score
    FROM orders o
    JOIN reviews r ON o.order_id = r.order_id
    WHERE o.order_status = 'delivered'
      AND o.order_delivered_customer_date IS NOT NULL
      AND r.review_score IS NOT NULL
""").df()

on_time = scores.loc[scores.delivery_status == "On time", "review_score"].to_numpy()
late = scores.loc[scores.delivery_status == "Late", "review_score"].to_numpy()


def cliffs_delta(x, y):
    """Cliff's delta via the Mann-Whitney U identity: d = 2U/(n*m) - 1."""
    u, _ = mannwhitneyu(x, y, alternative="two-sided")
    return 2.0 * u / (len(x) * len(y)) - 1.0


u_stat, p_value = mannwhitneyu(on_time, late, alternative="greater")
delta = cliffs_delta(on_time, late)

ad = abs(delta)
effect_size = ("negligible" if ad < 0.147 else "small" if ad < 0.33
               else "medium" if ad < 0.474 else "large")

# Bootstrap 95% CI for Cliff's delta.
rng = np.random.default_rng(42)
boot = np.array([
    cliffs_delta(rng.choice(on_time, len(on_time), replace=True),
                 rng.choice(late, len(late), replace=True))
    for _ in range(1000)
])
ci_low, ci_high = np.percentile(boot, [2.5, 97.5])

print(f"n on-time = {len(on_time):,}   n late = {len(late):,}")
print(f"median on-time = {np.median(on_time):.0f}   median late = {np.median(late):.0f}")
print(f"mean on-time   = {on_time.mean():.3f}   mean late   = {late.mean():.3f}")
print(f"Mann-Whitney U = {u_stat:,.0f}   p-value = {p_value:.3g}")
print(f"Cliff's delta  = {delta:.4f} ({effect_size}), "
      f"95% CI [{ci_low:.4f}, {ci_high:.4f}]")

n on-time = 88,653   n late = 7,700
median on-time = 5   median late = 2
mean on-time   = 4.294   mean late   = 2.566
Mann-Whitney U = 530,208,489   p-value = 0
Cliff's delta  = 0.5534 (large), 95% CI [0.5422, 0.5647]


In [16]:
# Persist A/B test results + the score distribution for the dashboard.
ab_result = pd.DataFrame({
    "metric": [
        "n_on_time", "n_late", "median_on_time", "median_late",
        "mean_on_time", "mean_late", "u_statistic", "p_value",
        "cliffs_delta", "cliffs_delta_ci_low", "cliffs_delta_ci_high",
        "effect_size",
    ],
    "value": [
        len(on_time), len(late),
        float(np.median(on_time)), float(np.median(late)),
        round(float(on_time.mean()), 3), round(float(late.mean()), 3),
        float(u_stat), p_value,
        round(delta, 4), round(ci_low, 4), round(ci_high, 4),
        effect_size,
    ],
})
ab_result.to_csv(EXPORT_DIR / "ab_test_result.csv", index=False)

ab_scores = con.execute("""
    SELECT
        CASE WHEN o.order_delivered_customer_date
                  > o.order_estimated_delivery_date
             THEN 'Late' ELSE 'On time' END AS delivery_status,
        r.review_score,
        COUNT(*) AS orders
    FROM orders o
    JOIN reviews r ON o.order_id = r.order_id
    WHERE o.order_status = 'delivered'
      AND o.order_delivered_customer_date IS NOT NULL
      AND r.review_score IS NOT NULL
    GROUP BY delivery_status, review_score
""").df()
ab_scores["pct_within_group"] = (
    ab_scores.groupby("delivery_status")["orders"]
    .transform(lambda s: round(s / s.sum() * 100, 1))
)
ab_scores = ab_scores.sort_values(["delivery_status", "review_score"])
ab_scores.to_csv(EXPORT_DIR / "ab_test_scores.csv", index=False)

ab_result

,metric,value
0,n_on_time,88653
1,n_late,7700
2,median_on_time,5.0
3,median_late,2.0
4,mean_on_time,4.294
5,mean_late,2.566
6,u_statistic,530208489.0
7,p_value,0.0
8,cliffs_delta,0.5534
9,cliffs_delta_ci_low,0.5422


## Conclusions

- **Revenue** grew from a near-zero 2016 ramp-up to ~R$ 1M/month through 2018,
  with a median MoM growth of ~7.8%; total delivered revenue ≈ R$ 15.4M.
- **Retention is very low**: only ~3% of customers (by `customer_unique_id`)
  ever place a second order — Olist behaves as a single-purchase marketplace.
- **RFM**: Champions (~24k customers) and At-Risk (~22k) together drive ~78% of
  revenue, so retaining/reactivating high-value buyers is where the upside is.
- **Delivery & reviews**: on-time orders average ~4.29 stars vs. ~2.57 for late
  orders. The A/B test confirms this is highly significant (Mann-Whitney U,
  p ≈ 0) with a **large** effect (Cliff's δ ≈ 0.55) — late delivery is the
  single biggest controllable driver of low review scores.

All result tables are written to `exports/` and rendered by the Streamlit app
(`app.py`).